# Day 2: Data Acquisition – Beyond Local Files

**Objective:** Acquire data from web sources (HTML tables, APIs, scraping) – essential for building real-world RAG datasets.

**Topics:**
- Reading HTML tables directly with `pd.read_html()`
- Calling public APIs with `requests` and loading JSON into DataFrames
- Web scraping with `BeautifulSoup` (static websites)
- Cleaning scraped data into a structured DataFrame

> 💡 **Memory aid**: Always check a website's `robots.txt` and terms of service before scraping. Use APIs whenever available.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

# Optional: for nicer display
from IPython.display import display, HTML

print("Libraries imported successfully.")

## 2. Reading HTML Tables with `pd.read_html()`

Many websites present data in HTML `<table>` elements. Pandas can extract all tables from a URL in one line.

In [ ]:
# Example: Wikipedia page of countries by GDP (nominal)
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

# read_html returns a list of DataFrames (one per table on the page)
tables = pd.read_html(url)
print(f"Found {len(tables)} tables on the page.")

In [ ]:
# Usually the main GDP table is the first or second table
# Let's inspect the first table
gdp_table = tables[0]
gdp_table.head()

In [ ]:
# Clean the table: remove footnote markers (e.g., [1]) from country names
# Simple cleaning: remove anything in brackets
import re
gdp_table['Country'] = gdp_table['Country'].apply(lambda x: re.sub(r'\[.*?\]', '', x).strip())
gdp_table.head()

**Exercise:** Try extracting the second table (often 'GDP per capita') and inspect its columns.

In [ ]:
# Your code here
# tables[1].head()


## 3. Calling a Public API with `requests`

APIs return structured data (usually JSON). We'll use a free weather API as an example.

In [ ]:
# Free API: Open-Meteo (no API key required)
# Get current weather for Bengaluru
lat, lon = 12.9716, 77.5946
api_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"

response = requests.get(api_url)
print(f"Status code: {response.status_code}")

# Parse JSON
weather_data = response.json()
weather_data

In [ ]:
# Convert JSON to DataFrame (if it's a simple dict, wrap in a list)
current = weather_data['current_weather']
df_weather = pd.DataFrame([current])
df_weather

In [ ]:
# More realistic: multiple cities
cities = [
    {'name': 'Bengaluru', 'lat': 12.9716, 'lon': 77.5946},
    {'name': 'Mumbai', 'lat': 19.0760, 'lon': 72.8777},
    {'name': 'Delhi', 'lat': 28.6139, 'lon': 77.2090}
]

weather_records = []
for city in cities:
    url = f"https://api.open-meteo.com/v1/forecast?latitude={city['lat']}&longitude={city['lon']}&current_weather=true"
    resp = requests.get(url).json()
    current_w = resp['current_weather']
    weather_records.append({
        'City': city['name'],
        'Temperature_C': current_w['temperature'],
        'Windspeed_kmh': current_w['windspeed'],
        'Weathercode': current_w['weathercode']
    })
    time.sleep(0.5)  # be polite to the free API

df_weather_multi = pd.DataFrame(weather_records)
df_weather_multi

**API Tip:** For APIs requiring authentication, store keys in a `.env` file and use `os.getenv('API_KEY')`.

## 4. Web Scraping with BeautifulSoup

When a website does not provide an API or direct CSV, we scrape HTML. We'll use `books.toscrape.com` – a safe, practice site.

In [ ]:
# Target: book titles, prices, and ratings from the first page
base_url = "http://books.toscrape.com/"
response = requests.get(base_url)
soup = BeautifulSoup(response.content, 'html.parser')
print(f"Page title: {soup.title.string}")

In [ ]:
# Find all book containers (each <article class="product_pod">)
books = soup.find_all('article', class_='product_pod')
print(f"Found {len(books)} books on page 1.")

In [ ]:
# Extract data
book_data = []
for book in books:
    title = book.h3.a['title']
    price = book.find('p', class_='price_color').text
    # Rating: class "star-rating X" where X = One, Two, Three, Four, Five
    rating_class = book.find('p', class_='star-rating')['class'][1]
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    rating = rating_map.get(rating_class, 0)
    book_data.append({
        'Title': title,
        'Price': price,
        'Rating': rating
    })

df_books = pd.DataFrame(book_data)
df_books.head()

In [ ]:
# Clean price column: remove £ and convert to float
df_books['Price'] = df_books['Price'].str.replace('£', '').astype(float)
df_books.info()

In [ ]:
# Average price by rating
df_books.groupby('Rating')['Price'].mean().round(2)

## 5. Handling Pagination (Scraping Multiple Pages)

In [ ]:
# Scrape first 3 pages of books.toscrape.com
all_books = []
for page in range(1, 4):  # pages 1 to 3
    url = f"http://books.toscrape.com/catalogue/page-{page}.html"
    print(f"Scraping {url}")
    resp = requests.get(url)
    soup = BeautifulSoup(resp.content, 'html.parser')
    books = soup.find_all('article', class_='product_pod')
    for book in books:
        title = book.h3.a['title']
        price = book.find('p', class_='price_color').text.replace('£', '')
        rating_class = book.find('p', class_='star-rating')['class'][1]
        rating = {'One':1, 'Two':2, 'Three':3, 'Four':4, 'Five':5}.get(rating_class, 0)
        all_books.append({'Title': title, 'Price': float(price), 'Rating': rating})
    time.sleep(0.5)  # be polite

df_all_books = pd.DataFrame(all_books)
print(f"Total books scraped: {len(df_all_books)}")
df_all_books.head()

## 6. Data Cleaning & Enrichment

Combine what we learned: clean scraped data, add derived columns.

In [ ]:
# Add price categories
def price_category(price):
    if price < 20:
        return 'Budget'
    elif price < 50:
        return 'Mid-range'
    else:
        return 'Premium'

df_all_books['PriceCategory'] = df_all_books['Price'].apply(price_category)
df_all_books['Rating'] = df_all_books['Rating'].astype('category')

# Summary by category
df_all_books.groupby('PriceCategory')['Title'].count()

## 7. Exercise: Scrape Financial News Headlines (for RAG later)

Try scraping a simple news site (e.g., Reuters headlines). Be respectful and check `robots.txt`.

In [ ]:
# Example: scraping BBC's business headlines (structure may change over time)
# This is a template – you may need to adjust selectors.

news_url = "https://www.bbc.com/news/business"
resp = requests.get(news_url)
soup = BeautifulSoup(resp.content, 'html.parser')

# Find all headline links (common pattern)
headlines = []
for item in soup.find_all('h3'):
    text = item.get_text(strip=True)
    if len(text) > 20:  # filter short texts
        headlines.append(text)

df_news = pd.DataFrame({'Headline': headlines[:10]})
df_news

**Note:** If the above does not return headlines (BBC may block scraping), try using `requests` with a User-Agent header or use a different practice site like `https://www.scrapethissite.com/`.

## 8. Putting It All Together – A Data Acquisition Pipeline

Here’s a function that fetches data from multiple sources and returns a cleaned DataFrame.

In [ ]:
def acquire_financial_data():
    """Demo: combines HTML table + API + scraped data"""
    # 1. GDP table
    gdp_df = pd.read_html("https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)")[0]
    gdp_df = gdp_df[['Country', 'GDP (US$million)']].rename(columns={'GDP (US$million)': 'GDP_M_USD'})
    gdp_df = gdp_df.head(5)
    
    # 2. Simple API: weather for each country's capital (mock)
    # For demo, we just add a placeholder
    gdp_df['Avg_Temp_C'] = [25, 22, 18, 20, 27]  # dummy
    
    return gdp_df

df_finance = acquire_financial_data()
df_finance

## Summary – Day 2 Completed ✅

You have learned:
- Extracting HTML tables with `pd.read_html()`
- Calling REST APIs and loading JSON into DataFrames
- Scraping static websites with `requests` + `BeautifulSoup`
- Handling pagination and cleaning scraped data

**Next:** Day 3 – Building a RAG Pipeline (loading, chunking, embeddings, vector store)

Save this notebook and commit to your repository.